# Assignment 1: Working with Spark DataFrames

# 1. Concept Check
* Explain the below in 3-5 sentences
* 5 points

## 1.1 Spark's distributed architecture and why is it so fast say as compared to Hadoop

* Spark's architecture consists of a Driver Program (which hosts the SparkContext), a Cluster Manager, and multiple Worker Nodes running executors that execute tasks and cache data in memory
. Spark is approximately 100x faster than Hadoop because Hadoop writes intermediate state to disk after every Map and Reduce phase, whereas Spark retains and processes data in memory (RAM) across a Directed Acyclic Graph (DAG) of operations
. It only spills to disk when memory limits require it



## 1.2 RDD Vs Data Frame Vs Data Set
* A Resilient Distributed Dataset (RDD) is Spark's core low-level abstraction representing an immutable, distributed collection of data elements
. DataFrames and DataSets are higher-level abstractions built on top of RDDs
. DataFrames organize data into named columns like a relational table for automatic query optimization, while DataSets add strongly typed object interfaces on top of tabular data

## 1.3 Spark Lazy Loading
* Spark uses lazy evaluation, meaning transformations applied to a dataset are not executed immediately when called
. Instead, Spark builds a Directed Acyclic Graph (DAG) tracking all the required transformations
. Actual execution is deferred until an action (such as count(), collect(), or writing output) is called, allowing Spark to optimize the full execution plan end-to-end

## 1.4 What is Delta and how does it help big data processing ?
* Delta is an open-source storage format built on top of Parquet files combined with an immutable transaction log (_delta_log)
. It aids big data processing by adding ACID transactional guarantees, schema evolution, and schema validation to data lakes
. Additionally, Delta uses its transaction log as a manifest to avoid expensive file listing operations, supports flexible UPSERTs, and unifies batch and streaming processing on a single data platform

## 1.5 What is data modeling and why should you care ?
* Data modeling is the practice of organizing, visualising, and structuring logical relationships in data to align with business processes and consumption patterns
. You should care because proper data modeling directly optimizes query performance, reduces data redundancy and storage costs, and enforces data quality constraints
. It also creates a shared vocabulary between technical and business teams so errors and inconsistencies are caught early

# 2. Read data

Put the data into a volume and set the {data-root} to the path of that volume.  You should have permission to do that in your workspace.  Here is documentation on how to create a volume: https://docs.databricks.com/aws/en/volumes/utility-commands
* Data set location: {data-root}/people10m
    * people10m-file1.parquet
* 10 points

In [0]:
# Drop if exist 

spark.sql('DROP CATALOG  IF exists  Assign1 CASCADE');

# create resources
spark.sql('CREATE CATALOG IF NOT EXISTS Assign1');
spark.sql('USE CATALOG Assign1')

spark.sql('CREATE SCHEMA IF NOT EXISTS Assign1');
spark.sql('USE Assign1.Assign1')

spark.sql('CREATE VOLUME IF NOT EXISTS input')

## 2.1 Read data in parquet format from given location ito a spark data frame

In [0]:
data_root = "/Volumes/Assign1/Assign1/input"
data_path = f"{data_root}/people10m-file1.parquet"
people_df = spark.read.parquet(data_path)
display(people_df.limit(5))

## 2.2 Examine Schema
* 1 point

In [0]:
people_df.printSchema()

## 2.3 Examine stats to 
* check data distribution of each column, 
* examine nulls, ranges etc
* 1 point

In [0]:
people_df.describe().show()

In [0]:
display(people_df.summary())

## 2.4 Select & Filter
* Use the DataFrame APIs select and filter methods
* select the following fields firstName,middleName,lastName,birthDate,gender
* filter rows where gender is Female/F and birth year > 1990
* 1 point

In [0]:
from pyspark.sql.functions import year

women = people_df.select(
    ['firstName', 'middleName','lastName','birthDate','gender']).filter(
    (people_df.gender == 'F') & (year(people_df.birthDate) > 1990)
)

display(women.count())

In [0]:
display(people_df.select("gender").distinct())

## 2.5 Query and Visualize
* How many women were named Mary in each year?
* Use an appropriate visualization to display your query result
* 1 point

In [0]:
mary_by_year_df = people_df.filter(
    (people_df.gender == 'F') & 
    (people_df.firstName == "Mary")
).groupBy(
    year(col("birthDate")).alias("birthYear")
).agg(
    count("*").alias("maryCount")
).orderBy("birthYear")

display(mary_by_year_df)

Databricks visualization. Run in Databricks to view.

## 2.6 Compare
* Compare popularity of two names from 1990 - Donna & Dorothy
* popularity is defined as #occurances in that year
* 1 point

In [0]:
compare_names_df = people_df.filter(
    (people_df.firstName.isin(['Donna', 'Dorothy'])) & 
    (year(people_df.birthDate) >= 1990)
).groupBy(
    "firstName"
).agg(
    count("*").alias("occurrences")
).orderBy("firstName")

display(compare_names_df)

## 2.7 Use agg & limit
* Create a DataFrame called top10FemaleFirstNamesDF that contains the 10 most common female first names out of the people data set.
* 1 point

In [0]:
from pyspark.sql.functions import count as count_func

top10FemaleFirstNamesDF = people_df.filter(
    people_df.gender == 'F'
).groupBy(
    "firstName"
).agg(
    count_func("*").alias("name_count")
).orderBy(
    col("name_count").desc()
).limit(10)

display(top10FemaleFirstNamesDF)

## 2.8 Create a temporary view & query it
* From the original dataframe of all the data
* Run the same query from before this time using ql and compare counts - How many women were named Mary in each year?
* 1 point

In [0]:
people_df.createOrReplaceTempView("people_view")

mary_sql_df = spark.sql("""
    SELECT 
        YEAR(birthDate) as birthYear,
        COUNT(*) as maryCount
    FROM people_view
    WHERE gender = 'F' AND firstName = 'Mary'
    GROUP BY YEAR(birthDate)
    ORDER BY birthYear
""")

display(mary_sql_df)

## 2.9 Persist data on disk
* Create a database <your user name>_Assignment1
* Persist all the data in delta format into a table called people_delta
* 1 point

In [0]:
spark.sql('CREATE CATALOG IF NOT EXISTS cscie103_catalog')
spark.sql('USE CATALOG cscie103_catalog')
spark.sql('CREATE SCHEMA IF NOT EXISTS assignment_01')
spark.sql('USE cscie103_catalog.assignment_01')

In [0]:
people_df.write.format("delta").mode("overwrite").saveAsTable("cscie103_catalog.assignment_01.people_delta")

## 2.10 Read data formats
* Read data from people_delta table
* Display schema
* Check counts on both
* Calculate average salary & min/max salary using built-in functions
* 1 point

In [0]:
from pyspark.sql.functions import avg, min as min_func, max as max_func
delta_df = spark.read.table("cscie103_catalog.assignment_01.people_delta")

delta_df.printSchema()

# Check counts
original_count = people_df.count()
delta_count = delta_df.count()
print(f"Original DataFrame Count: {original_count}")
print(f"Delta Table Count:        {delta_count}")

# Calculate salary statistics
salary_summary_df = delta_df.select(
    avg("salary").alias("avg_salary"),
    min_func("salary").alias("min_salary"),
    max_func("salary").alias("max_salary")
)

display(salary_summary_df)

# 3. Joins 
* 5 points

## 3.1 Load data 
* File will be on google drive in https://drive.google.com/drive/folders/1Wps0LoyMWNc00g7GLpk_UaARCdNxBBz5?usp=drive_link
* Data set location: {data-root}/names-1880-2024
  * names_file1.parquet
* Perform a distinct count on first name on both data frames
* 1 point

In [0]:
# Load names dataset from parquet
names_path = f"{data_root}/names_file1.parquet"
names_df = spark.read.parquet(names_path)

display(names_df.limit(5))

In [0]:
# Perform distinct count on firstName for both dataframes
people_distinct = people_df.select("firstName").distinct().count()
names_distinct = names_df.select("firstName").distinct().count()

print(f"Distinct first names in people_df: {people_distinct}")
print(f"Distinct first names in names_df:  {names_distinct}")

## 3.2 Inner Join 
* the 2 dataframes on first name
* 1 point

In [0]:
# Inner join on firstName
# Alias DataFrames and select specific columns to avoid ambiguity
people_aliased = people_df.alias("people")
names_aliased = names_df.alias("names")

joined_df = people_aliased.join(
    names_aliased,
    people_aliased.firstName == names_aliased.firstName,
    "inner"
).select(
    col("people.firstName"),
    col("people.id").alias("people_id"),
    col("people.middleName"),
    col("people.lastName"),
    col("people.gender").alias("people_gender"),
    col("names.gender").alias("names_gender"),
    col("people.birthDate"),
    col("people.ssn"),
    col("people.salary"),
    col("names.year"),
    col("names.total")
)

print(f"Joined DataFrame count: {joined_df.count()}")
display(joined_df.limit(5))


## 3.3 Data Cleanse
* Convert negative salaries to positive numbers
* 1 point

In [0]:
from pyspark.sql.functions import abs as abs_func

cleansed_df = joined_df.withColumn("salary", abs_func(col("salary")))

print(f"Cleansed DataFrame count: {cleansed_df.count()}")
display(cleansed_df.select("firstName", "salary").limit(10))

## 3.3 Data Transform
* assume all salaries under $20,000 represent bad rows and filter them out.
* categorize each person's salary into $10K groups.
  * A salary of 23,000 should report a value of "2".
  * A salary of 57,400 should report a value of "6".
* 1 point

In [0]:
from pyspark.sql.functions import round

transformed_df = cleansed_df.filter(
    col("salary") >= 20000
).withColumn(
    "salary_category", 
    round(col("salary") / 10000).cast("int")
)

print(f"Transformed DataFrame count: {transformed_df.count()}")
display(transformed_df.select("firstName", "salary", "salary_category").orderBy("salary").limit(10))

## 3.4 Concept Q
* Write the difference betweem the various join options
* Can you join on multiple conditions
* 1 point

## Join Types in PySpark

### Different Join Options:

**1. Inner Join:**
* Returns only matching rows present in both DataFrames
* Rows without a match in either DataFrame are excluded

**2. Left Outer Join \:**
* Retains all rows from the left DataFrame
* Attaches matching rows from the right DataFrame
* Populates null for non-matches from the right side

**3. Right Outer Join:**
* Retains all rows from the right DataFrame
* Attaches matching rows from the left DataFrame
* Populates null for non-matches from the left side

**4. Full Outer Join:**
* Retains all rows from both DataFrames
* Populates null where keys do not match on either side

**5. Left Semi Join:**
* Evaluates matches like an inner join
* Returns only columns from the left DataFrame for rows that have a match in the right DataFrame
* Similar to an SQL EXISTS subquery

**6. Left Anti Join (`"left_anti"`):**
* Returns only rows from the left DataFrame that do NOT have a matching key in the right DataFrame
* Similar to an SQL NOT EXISTS subquery

**7. Cross Join (`"cross"`):**
* Produces a Cartesian product
* Combines every row from the left DataFrame with every row from the right DataFrame
* Use with caution - can create very large result sets

---

### Joining on Multiple Conditions:

**Yes**, you can join on multiple conditions in PySpark using two approaches:

**Approach 1: List of column names (for equality conditions)**
```python
df1.join(df2, on=["firstName", "lastName"], how="inner")
```

**Approach 2: Logical expressions (for complex conditions)**
```python
df1.join(
    df2,
    (df1.firstName == df2.firstName) & (df1.birthYear == df2.year),
    how="inner"
)
```

* Use & for AND conditions
* Use | for OR conditions
* Always wrap each condition in parentheses when combining

# 4. Semi-Structured Data
* Ex. JSON, XML
* 10 points

## 4.1 Read JSON

* File will be on google drive in https://drive.google.com/drive/folders/1Wps0LoyMWNc00g7GLpk_UaARCdNxBBz5?usp=drive_link
* Data set location: {data-root}
  * databricks-blog.json
* PrintSchema
* 1 point

In [0]:
blog_path = f"{data_root}/databricks-blog.json"
blog_df = spark.read.option("multiLine", "true").json(blog_path)

blog_df.printSchema()
display(blog_df.limit(5))

## 4.2 Access fields
* author, categories, title
* nested date fields individually
* Hint: select("dates.publishedOn")
* add a new column publishedOn at the top level and populate it from the nested date sub-field
* 1 point

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StructField, StringType

# Define schema for dates JSON string
dates_schema = StructType([
    StructField("createdOn", StringType(), True),
    StructField("publishedOn", StringType(), True),
    StructField("tz", StringType(), True)
])

# Parse dates JSON string and access nested fields
accessed_fields_df = blog_df.select(
    "authors",
    "categories",
    "title",
    from_json(col("dates"), dates_schema).alias("dates_parsed")
).select(
    "authors",
    "categories",
    "title",
    col("dates_parsed.publishedOn").alias("publishedOn")
)

display(accessed_fields_df)


## 4.3 Date time fields
* cast publishedOn to timestamp using date_format
* filter on published year of 2013
* 1 point

In [0]:

from pyspark.sql.functions import  to_timestamp, year

dated_df = accessed_fields_df.withColumn(
    "publishedOnTimestamp", to_timestamp(col("publishedOn"))
).filter(
    year(col("publishedOnTimestamp")) == 2013
)

display(dated_df)

## 4.4 Array data
* Select the first author in the array aauthors as the primaryAuthor
* Hint: select("col(authors)[0]")
* 1 point

In [0]:
from pyspark.sql.types import ArrayType

authors_schema = ArrayType(StringType())
primary_author_df = blog_df.select(
    "title",
    from_json(col("authors"), authors_schema).getItem(0).alias("primaryAuthor")
)

display(primary_author_df)


## 4.5 Explode
* Split single row to multiple fields (for authors)
* Show title & each author on different rows
* Hint: select(explode(col("authors")).alias("Author")
* 1 point

In [0]:
from pyspark.sql.functions import explode, from_json

authors_schema = ArrayType(StringType())

exploded_authors_df = blog_df.select(
    "title",
    explode(from_json(col("authors"), authors_schema)).alias("Author")
)

display(exploded_authors_df)

## 4.6 Search
* Identify all the articles written or co-written by Michael Armbrust.
* 1 point

In [0]:
from pyspark.sql.functions import  array_contains

authors_schema = ArrayType(StringType())

michael_articles_df = blog_df.filter(
    array_contains(from_json(col("authors"), authors_schema), "Michael Armbrust")
).select("title", "authors")

display(michael_articles_df)

## 4.7 Count
* Count how many times each category is referenced in the Databricks blog.
* 2 point

In [0]:

categories_schema = ArrayType(StringType())

category_counts_df = blog_df.select(
    explode(from_json(col("categories"), categories_schema)).alias("category")
).groupBy("category").agg(
    count("*").alias("referenceCount")
).orderBy(col("referenceCount").desc())

display(category_counts_df)


## 4.8 Concept Q: 
* How will you handle multi-line json data
* 1 point

By default, Spark expects single-line JSON objects (JSON Lines format). To read a multi-line JSON document where a single object spans multiple lines, set the multiLine option to True
:

## 4.9 Concept Q:
* If your column contains json data, how will you parse it (which function) while reading
* How will you save data as json
* 1 point

* Parsing JSON inside a Column: Use the from_json(col, schema) function by supplying the JSON string column along with a defined StructType or ArrayType schema
. 
* Alternatively, get_json_object(col, path) can extract specific fields using JSONPath syntax.
Saving Data as JSON: Use df.write.json("output_path") or df.write.format("json").save("output_path") to persist the DataFrame as JSON files
.